In [ ]:
!pip install -q datasets transformers evaluate scikit-learn accelerate seaborn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.0 MB/s eta 0:00:00


In [4]:
!pip freeze | grep -iE "^(datasets|transformers|torch|scikit-learn|pandas|numpy|accelerate|evaluate|matplotlib|seaborn)=="

accelerate==1.15.0
datasets==4.8.5
matplotlib==3.10.0
numpy==2.1.3
pandas==2.2.3
scikit-learn==1.6.1
seaborn==0.13.2
transformers==5.20.1


In [ ]:
!pip freeze | grep -iE "^(datasets|transformers|torch|scikit-learn|pandas|numpy|accelerate|evaluate|matplotlib|seaborn)=="

accelerate==1.15.0
datasets==4.8.5
matplotlib==3.10.0
numpy==2.1.3
pandas==2.2.3
scikit-learn==1.6.1
seaborn==0.13.2
transformers==5.20.1


In [ ]:
import torch
print(torch.cuda.is_available())

True


In [ ]:
from datasets import load_dataset

ds = load_dataset("HausaNLP/NaijaSenti-Twitter", "pcm")
print(ds)
print(ds["train"][0])

pcm/train/0000.parquet: reconstructing file:   0%|          |  0.00B /  407kB            

pcm/train/0000.parquet: downloading bytes:           |  0.00B            

pcm/validation/0000.parquet: reconstructing file:   0%|          |  0.00B /  105kB            

pcm/validation/0000.parquet: downloading bytes:           |  0.00B            

pcm/test/0000.parquet: reconstructing file:   0%|          |  0.00B /  276kB            

pcm/test/0000.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/5121 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1281 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/4154 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['tweet', 'label'],
        num_rows: 5121
    })
    validation: Dataset({
        features: ['tweet', 'label'],
        num_rows: 1281
    })
    test: Dataset({
        features: ['tweet', 'label'],
        num_rows: 4154
    })
})
{'tweet': 'yeah \u200d️the guy wants to trend dat was why e join nysc e cant trend with good music again', 'label': 2}


In [ ]:
!pip install -q "datasets<4"

In [ ]:
import pandas as pd

train_df = ds["train"].to_pandas()
val_df = ds["validation"].to_pandas()
test_df = ds["test"].to_pandas()

print(train_df.shape, val_df.shape, test_df.shape)
print(train_df.head())
print(train_df["label"].value_counts())

(5121, 2) (1281, 2) (4154, 2)
                                               tweet  label
0  yeah ‍️the guy wants to trend dat was why e jo...      2
1  this life is so funny sef you will work hard a...      2
2  dis is unfair of urcompany goingtowks nw ive b...      2
3  lil wayne don vex me im actually not excited a...      2
4  dis is unfair of ur company going to wks nw iv...      2
label
2    3241
0    1808
1      72
Name: count, dtype: int64


In [ ]:
print(ds["train"].features["label"])

print("Duplicate tweets in train:", train_df["tweet"].duplicated().sum())

overlap = set(train_df["tweet"]) & set(test_df["tweet"])
print("Tweets in both train and test:", len(overlap))

ClassLabel(names=['positive', 'neutral', 'negative'], id=None)
Duplicate tweets in train: 512
Tweets in both train and test: 926


In [ ]:
import re

def clean(text):
    text = re.sub(r"http\S+", "<url>", text)
    text = re.sub(r"@\w+", "<user>", text)
    return text.strip().lower()

for df in (train_df, val_df, test_df):
    df["clean"] = df["tweet"].apply(clean)

conflicts = train_df.groupby("clean")["label"].nunique().gt(1).sum()
print("Same tweet, different labels:", conflicts)

before = (len(train_df), len(val_df), len(test_df))

train_df = train_df.drop_duplicates(subset="clean").reset_index(drop=True)

val_df = val_df.drop_duplicates(subset="clean")
val_df = val_df[~val_df["clean"].isin(train_df["clean"])].reset_index(drop=True)

seen = set(train_df["clean"]) | set(val_df["clean"])
test_df = test_df.drop_duplicates(subset="clean")
test_df = test_df[~test_df["clean"].isin(seen)].reset_index(drop=True)

print("Before:", before)
print("After:", (len(train_df), len(val_df), len(test_df)))
print(train_df["label"].value_counts())
print(test_df["label"].value_counts())

Same tweet, different labels: 0
Before: (4609, 996, 3026)
After: (4609, 996, 3026)
label
2    3003
0    1540
1      66
Name: count, dtype: int64
label
2    1572
0    1025
1     429
Name: count, dtype: int64


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score

tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
X_train = tfidf.fit_transform(train_df["clean"])
X_val = tfidf.transform(val_df["clean"])
X_test = tfidf.transform(test_df["clean"])

clf = LogisticRegression(max_iter=1000, class_weight="balanced")
clf.fit(X_train, train_df["label"])

names = ["positive", "neutral", "negative"]
for split, X, df in [("VALIDATION", X_val, val_df), ("TEST", X_test, test_df)]:
    preds = clf.predict(X)
    print(split, "macro-F1:", round(f1_score(df["label"], preds, average="macro"), 3))
    print(classification_report(df["label"], preds, target_names=names))

VALIDATION macro-F1: 0.426
              precision    recall  f1-score   support

    positive       0.52      0.53      0.53       331
     neutral       0.00      0.00      0.00        17
    negative       0.75      0.76      0.75       648

    accuracy                           0.67       996
   macro avg       0.42      0.43      0.43       996
weighted avg       0.66      0.67      0.66       996

TEST macro-F1: 0.433
              precision    recall  f1-score   support

    positive       0.57      0.58      0.57      1025
     neutral       0.29      0.00      0.01       429
    negative       0.65      0.81      0.72      1572

    accuracy                           0.62      3026
   macro avg       0.50      0.46      0.43      3026
weighted avg       0.57      0.62      0.57      3026



In [ ]:
import numpy as np, torch
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)

MODEL = "xlm-roberta-base"
tok = AutoTokenizer.from_pretrained(MODEL)

def to_hf(df):
    d = Dataset.from_pandas(df[["clean", "label"]].rename(columns={"label": "labels"}), preserve_index=False)
    return d.map(lambda b: tok(b["clean"], truncation=True, max_length=128), batched=True)

hf_train, hf_val, hf_test = to_hf(train_df), to_hf(val_df), to_hf(test_df)

def metrics(p):
    preds = np.argmax(p.predictions, axis=-1)
    return {"macro_f1": f1_score(p.label_ids, preds, average="macro")}

model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=3)
args = TrainingArguments(output_dir="out", num_train_epochs=3,
    per_device_train_batch_size=16, per_device_eval_batch_size=32,
    learning_rate=2e-5, weight_decay=0.01, eval_strategy="epoch",
    save_strategy="no", seed=1, fp16=torch.cuda.is_available(), report_to="none")

trainer = Trainer(model=model, args=args, train_dataset=hf_train,
    eval_dataset=hf_val, data_collator=DataCollatorWithPadding(tok),
    compute_metrics=metrics)
trainer.train()

out = trainer.predict(hf_test)
t_preds = np.argmax(out.predictions, axis=-1)
print("TEST macro-F1:", round(f1_score(test_df["label"], t_preds, average="macro"), 3))
print(classification_report(test_df["label"], t_preds, target_names=names))

Map:   0%|          | 0/4609 [00:00<?, ? examples/s]

Map:   0%|          | 0/996 [00:00<?, ? examples/s]

Map:   0%|          | 0/3026 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Macro F1
1,No log,0.650275,0.451617
2,0.681434,0.635839,0.448536
3,0.681434,0.640087,0.462916


TEST macro-F1: 0.446
              precision    recall  f1-score   support

    positive       0.63      0.56      0.59      1025
     neutral       0.00      0.00      0.00       429
    negative       0.65      0.88      0.75      1572

    accuracy                           0.65      3026
   macro avg       0.43      0.48      0.45      3026
weighted avg       0.55      0.65      0.59      3026



/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
scores, all_preds = [], {}
for seed in [1, 2, 3]:
    model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=3)
    args = TrainingArguments(output_dir=f"out{seed}", num_train_epochs=3,
        per_device_train_batch_size=16, per_device_eval_batch_size=32,
        learning_rate=2e-5, weight_decay=0.01, eval_strategy="no",
        save_strategy="no", seed=seed, fp16=torch.cuda.is_available(),
        report_to="none")
    tr = Trainer(model=model, args=args, train_dataset=hf_train,
        data_collator=DataCollatorWithPadding(tok))
    tr.train()
    p = np.argmax(tr.predict(hf_test).predictions, axis=-1)
    all_preds[seed] = p
    scores.append(f1_score(test_df["label"], p, average="macro"))
    print("seed", seed, "macro-F1:", round(scores[-1], 3))

print("XLM-R mean:", round(np.mean(scores), 3), "+/-", round(np.std(scores), 3))

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
500,0.683145


seed 1 macro-F1: 0.443


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

seed 2 macro-F1: 0.407


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
500,0.698117


seed 3 macro-F1: 0.416
XLM-R mean: 0.422 +/- 0.015


In [ ]:
names = ["positive", "neutral", "negative"]
err = test_df.copy()
err["pred"] = all_preds[1]
err["true_name"] = err["label"].map(dict(enumerate(names)))
err["pred_name"] = err["pred"].map(dict(enumerate(names)))
wrong = err[err["label"] != err["pred"]]
print("Wrong:", len(wrong), "of", len(err))

print("\n--- 12 NEUTRAL tweets from test ---")
for t in err[err["label"] == 1]["tweet"].sample(12, random_state=1):
    print("-", t[:150])

sample = wrong.sample(50, random_state=42)[["tweet", "true_name", "pred_name"]]
sample["error_type"] = ""
sample.to_csv("error_analysis.csv", index=False)

from google.colab import files
files.download("error_analysis.csv")

Wrong: 1085 of 3026

--- 12 NEUTRAL tweets from test ---
- oga u don dey port
- i get work wey i dey do that side wan prime on tuesday then run asphalt next week
- word with jennifer omoko weti bird see chop nai e take dey hammer or nai e go take hammer weti you understand by this proverb
- nor be una dey find proof
- i still dey reason wetin better days dey do ahead
- mk i dey look you go na aisha dey hold my hand
- na we dey rush our phone
- nah we were both in a hurry it was yrs ago sef i remembercus it was a day to oshuns wedding went to pick up my suit den
- na the same anambra wey i dey o
- for this one wey una don talk so i dey resume tomorrow abi evening session dey today
- na movement una dey like this oo
- na wetin people wey get money dey always talk


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
markers = r"\b(dey|wey|na|wetin|una|abi|oga|sef|nor|don|wan|nah)\b"
test_df["pidgin_words"] = test_df["clean"].str.count(markers)
test_df["words"] = test_df["clean"].str.split().str.len()
test_df["pidgin_rate"] = test_df["pidgin_words"] / test_df["words"]
print(test_df.groupby("label")["pidgin_rate"].mean().round(3))

print("\n--- 25 mistakes ---")
for _, r in wrong.sample(25, random_state=7).iterrows():
    print(r["true_name"], "->", r["pred_name"], "|", r["tweet"][:140])

label
0    0.161
1    0.224
2    0.159
Name: pidgin_rate, dtype: float64

--- 25 mistakes ---
neutral -> negative | mad oo you for tell them wetin you hear na
neutral -> negative | karim wey dey chop pie creampie
neutral -> positive | my guy dey de list o
neutral -> negative | oluwa wetin dey occur
negative -> positive | wetin dey worry this one i dey find who go front for me join giveaway you dey match me bill
neutral -> positive | please get started o he is coming to mt club o make i sabi as e dey go i can fast and pray o
negative -> positive | you wey dey post prick u sabi bible join
positive -> negative | na so you sabi wetin time dey talk
neutral -> negative | no be same place abina wetin again
neutral -> negative | anyone with like million wey e no dey use will give you back in
neutral -> positive | i still dey reason wetin better days dey do ahead
positive -> negative | i dey enerhen junction nothing dey happen
positive -> negative | na i wan dey discuss ideas with now na wetin 

In [ ]:
print(pd.crosstab(err["true_name"], err["pred_name"]))

sample = pd.read_csv("error_analysis.csv")
print()
for i, r in sample.iterrows():
    print(i + 1, "|", r["true_name"], "->", r["pred_name"], "|", r["tweet"][:130])

NameError: name 'err' is not defined

In [ ]:
import re, numpy as np, pandas as pd, torch
from datasets import load_dataset, Dataset
from sklearn.metrics import f1_score
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)

ds = load_dataset("HausaNLP/NaijaSenti-Twitter", "pcm", trust_remote_code=True)
train_df = ds["train"].to_pandas()
val_df = ds["validation"].to_pandas()
test_df = ds["test"].to_pandas()

def clean(t):
    t = re.sub(r"http\S+", "<url>", t)
    t = re.sub(r"@\w+", "<user>", t)
    return t.strip().lower()

for d in (train_df, val_df, test_df):
    d["clean"] = d["tweet"].apply(clean)

train_df = train_df.drop_duplicates(subset="clean").reset_index(drop=True)
val_df = val_df.drop_duplicates(subset="clean")
val_df = val_df[~val_df["clean"].isin(train_df["clean"])].reset_index(drop=True)
seen = set(train_df["clean"]) | set(val_df["clean"])
test_df = test_df.drop_duplicates(subset="clean")
test_df = test_df[~test_df["clean"].isin(seen)].reset_index(drop=True)
print("Sizes:", len(train_df), len(val_df), len(test_df))

MODEL = "xlm-roberta-base"
tok = AutoTokenizer.from_pretrained(MODEL)

def to_hf(d):
    x = Dataset.from_pandas(d[["clean", "label"]].rename(columns={"label": "labels"}), preserve_index=False)
    return x.map(lambda b: tok(b["clean"], truncation=True, max_length=128), batched=True)

hf_train, hf_test = to_hf(train_df), to_hf(test_df)

model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=3)
args = TrainingArguments(output_dir="out", num_train_epochs=3,
    per_device_train_batch_size=16, per_device_eval_batch_size=32,
    learning_rate=2e-5, weight_decay=0.01, eval_strategy="no",
    save_strategy="no", seed=1, fp16=torch.cuda.is_available(), report_to="none")
tr = Trainer(model=model, args=args, train_dataset=hf_train,
             data_collator=DataCollatorWithPadding(tok))
tr.train()

pred = np.argmax(tr.predict(hf_test).predictions, axis=-1)
print("TEST macro-F1:", round(f1_score(test_df["label"], pred, average="macro"), 3))

names = ["positive", "neutral", "negative"]
err = test_df.copy()
err["pred"] = pred
err["true_name"] = err["label"].map(dict(enumerate(names)))
err["pred_name"] = err["pred"].map(dict(enumerate(names)))
wrong = err[err["label"] != err["pred"]]
print("Wrong:", len(wrong), "of", len(err))

print()
print(pd.crosstab(err["true_name"], err["pred_name"]))

sample = wrong.sample(50, random_state=42)
print()
for i, (_, r) in enumerate(sample.iterrows(), start=1):
    print(i, "|", r["true_name"], "->", r["pred_name"], "|", r["tweet"][:130])

FileNotFoundError: Unable to find 'hf://datasets/HausaNLP/NaijaSenti-Twitter@main/pcm/train-00000-of-00001.parquet'

In [ ]:
import re, numpy as np, pandas as pd, torch
from datasets import load_dataset, Dataset
from sklearn.metrics import f1_score
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)

try:
    ds = load_dataset("HausaNLP/NaijaSenti-Twitter", "pcm", trust_remote_code=True)
except Exception as e:
    print("First method failed:", str(e)[:150])
    ds = load_dataset("HausaNLP/NaijaSenti-Twitter", "pcm", revision="refs/convert/parquet")
print(ds)

train_df = ds["train"].to_pandas()
val_df = ds["validation"].to_pandas()
test_df = ds["test"].to_pandas()

def clean(t):
    t = re.sub(r"http\S+", "<url>", t)
    t = re.sub(r"@\w+", "<user>", t)
    return t.strip().lower()

for d in (train_df, val_df, test_df):
    d["clean"] = d["tweet"].apply(clean)

train_df = train_df.drop_duplicates(subset="clean").reset_index(drop=True)
val_df = val_df.drop_duplicates(subset="clean")
val_df = val_df[~val_df["clean"].isin(train_df["clean"])].reset_index(drop=True)
seen = set(train_df["clean"]) | set(val_df["clean"])
test_df = test_df.drop_duplicates(subset="clean")
test_df = test_df[~test_df["clean"].isin(seen)].reset_index(drop=True)
print("Sizes:", len(train_df), len(val_df), len(test_df))

MODEL = "xlm-roberta-base"
tok = AutoTokenizer.from_pretrained(MODEL)

def to_hf(d):
    x = Dataset.from_pandas(d[["clean", "label"]].rename(columns={"label": "labels"}), preserve_index=False)
    return x.map(lambda b: tok(b["clean"], truncation=True, max_length=128), batched=True)

hf_train, hf_test = to_hf(train_df), to_hf(test_df)

model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=3)
args = TrainingArguments(output_dir="out", num_train_epochs=3,
    per_device_train_batch_size=16, per_device_eval_batch_size=32,
    learning_rate=2e-5, weight_decay=0.01, eval_strategy="no",
    save_strategy="no", seed=1, fp16=torch.cuda.is_available(), report_to="none")
tr = Trainer(model=model, args=args, train_dataset=hf_train,
             data_collator=DataCollatorWithPadding(tok))
tr.train()

pred = np.argmax(tr.predict(hf_test).predictions, axis=-1)
print("TEST macro-F1:", round(f1_score(test_df["label"], pred, average="macro"), 3))

names = ["positive", "neutral", "negative"]
err = test_df.copy()
err["pred"] = pred
err["true_name"] = err["label"].map(dict(enumerate(names)))
err["pred_name"] = err["pred"].map(dict(enumerate(names)))
wrong = err[err["label"] != err["pred"]]
print("Wrong:", len(wrong), "of", len(err))

print()
print(pd.crosstab(err["true_name"], err["pred_name"]))

sample = wrong.sample(50, random_state=42)
print()
for i, (_, r) in enumerate(sample.iterrows(), start=1):
    print(i, "|", r["true_name"], "->", r["pred_name"], "|", r["tweet"][:130])

pcm/train/0000.parquet: reconstructing file:   0%|          |  0.00B /  407kB            

pcm/train/0000.parquet: downloading bytes:           |  0.00B            

pcm/validation/0000.parquet: reconstructing file:   0%|          |  0.00B /  105kB            

pcm/validation/0000.parquet: downloading bytes:           |  0.00B            

pcm/test/0000.parquet: reconstructing file:   0%|          |  0.00B /  276kB            

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
500,0.675556


TEST macro-F1: 0.442
Wrong: 1093 of 3026

pred_name  negative  positive
true_name                    
negative       1364       208
neutral         292       137
positive        456       569

1 | neutral -> negative | shey na wey dey here
2 | positive -> negative | them go dey whatsapp group na wetin dem say make them dey preach
3 | positive -> negative | but e reveal to you say una go beat wolves na
4 | neutral -> negative | how far na how u dey shey rain don dey fall for ur side
5 | neutral -> positive | we plenty o me sef dey wait for my own supply
6 | neutral -> negative | wetin de ballet
7 | positive -> negative | andprf cletus una gudevenssg i tink it should be bb bonsnegodu liverpool negodu messii dey thnk u joor
8 | positive -> negative | i dey warri oooooo who dey warri
9 | negative -> positive | lmao stop acting like all these things nor dey sweet you dey e dey turn you on boss
10 | neutral -> positive | i still de find team wey fit break this record oooo i neva see
11 | neg

In [ ]:
print(pd.crosstab(err["true_name"], err["pred_name"]))

sample = wrong.sample(50, random_state=42)
print()
for i, (_, r) in enumerate(sample.iterrows(), start=1):
    print(i, "|", r["true_name"], "->", r["pred_name"], "|", r["tweet"][:130])

NameError: name 'err' is not defined

In [ ]:

import datasets
print(datasets.__version__)

3.6.0


In [ ]:
!pip install -q "datasets<4" transformers evaluate scikit-learn accelerate seaborn